# Workshop 8 — Streaming: Make the Assistant Feel Real-Time with NVIDIA NIM

Workshop 7's assistant waits for the *entire* answer, then prints it at once. For a one-line reply that's fine; for a paragraph it hangs for a few seconds and then dumps text. **Streaming** fixes that — print each token the moment the model generates it, the way ChatGPT does. That single change is what makes an assistant feel like a product instead of a script.

The flag is trivial: `stream=True`. The lesson is what the stream gives back. A streamed response arrives as many small **chunks**, not one message:

- Text arrives as `delta.content` pieces you concatenate.
- Tool calls arrive **split across chunks** — the name and the arguments JSON come in fragments identified by an `index` — so you reassemble them by index before running them.

The agent loop itself does **not** change. Streaming is a data-accumulation layer on top of the exact Workshop 7 loop. Same hosted endpoint, same `meta/llama-3.3-70b-instruct`, same three tools.

## Step 1 — Setup (self-contained)

Carries the retriever from Workshop 2 and the three tools from Workshop 6. Paste your NVIDIA API key when prompted.

In [ ]:
%pip install -q openai numpy

import os, getpass, json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from openai import OpenAI
import numpy as np

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

client = OpenAI(
    base_url='https://integrate.api.nvidia.com/v1',
    api_key=os.environ['NVIDIA_API_KEY'],
)

MODEL = 'meta/llama-3.3-70b-instruct'
EMBED_MODEL = 'nvidia/nv-embedqa-e5-v5'
LOCAL_TZ = 'America/Los_Angeles'

knowledge_base = [
    {'title': 'USC AI Club meeting',
     'text': 'The USC AI Club meets every Thursday at 5 PM in the engineering building, room 204.'},
    {'title': 'USC GPU lab hours',
     'text': 'The USC GPU computing lab is open Monday to Friday from 10 AM to 6 PM.'},
    {'title': 'NVIDIA Developer Program',
     'text': 'USC students can join the NVIDIA Developer Program for free.'},
    {'title': 'Next USC workshop',
     'text': 'The next USC AI Club workshop will cover Retrieval Augmented Generation (RAG).'},
    {'title': 'USC AI/ML office hours',
     'text': 'Office hours for the USC AI/ML faculty are Tuesdays 2-4 PM.'},
    {'title': 'USC robotics lab',
     'text': 'The USC robotics lab requires safety training before students can use the soldering station.'},
    {'title': 'USC tutoring',
     'text': 'Peer tutoring for introductory Python at USC is available Wednesdays from 1 PM to 3 PM.'},
]

def embed_texts(texts, input_type='passage'):
    response = client.embeddings.create(model=EMBED_MODEL, input=texts, extra_body={'input_type': input_type})
    return [np.array(item.embedding, dtype=np.float32) for item in response.data]

def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

def retrieve_context(question, k=3):
    q_emb = embed_texts([question], input_type='query')[0]
    scored = [(cosine_similarity(q_emb, item['embedding']), item) for item in knowledge_base]
    scored.sort(key=lambda p: p[0], reverse=True)
    return '\n'.join(f"- {item['text']}" for _, item in scored[:k])

for item, emb in zip(knowledge_base, embed_texts([i['text'] for i in knowledge_base], 'passage')):
    item['embedding'] = emb
print(f'Ready. Embedded {len(knowledge_base)} chunks. Model: {MODEL}')

## Step 2 — The three tools from Workshop 6 (unchanged)

In [ ]:
WEEKDAYS = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

def get_current_time(timezone=LOCAL_TZ):
    try:
        zone = ZoneInfo(timezone)
    except Exception:
        zone = ZoneInfo('UTC')
    return datetime.now(zone).strftime('%A, %B %d, %Y at %I:%M %p %Z')

def search_campus_info(query):
    return retrieve_context(query, k=3)

def days_until_weekday(weekday):
    target = weekday.strip().capitalize()
    if target not in WEEKDAYS:
        return f"'{weekday}' is not a valid weekday. Use one of: {', '.join(WEEKDAYS)}."
    today = datetime.now(ZoneInfo(LOCAL_TZ))
    delta = (WEEKDAYS.index(target) - today.weekday()) % 7
    date_str = (today + timedelta(days=delta)).strftime('%B %d, %Y')
    if delta == 0:
        return f"Today is {target} ({date_str}) — that is 0 days away (it's today)."
    return f"The next {target} is in {delta} day(s), on {date_str}. Today is {today.strftime('%A')}."

tools = [
    {'type': 'function', 'function': {
        'name': 'search_campus_info',
        'description': 'Search the USC campus knowledge base for facts about USC clubs (including the AI Club), labs (GPU lab, robotics lab), workshops, faculty office hours, peer tutoring, and the NVIDIA Developer Program. Use this to find WHEN or WHERE something happens. Always call this for any USC-specific fact — do not answer from your own knowledge.',
        'parameters': {'type': 'object',
            'properties': {'query': {'type': 'string', 'description': 'The USC campus question or search phrase.'}},
            'required': ['query']},
    }},
    {'type': 'function', 'function': {
        'name': 'get_current_time',
        'description': 'Get the current date, day of week, and time in an IANA time zone. Use this when the question depends on what day or time it is right now.',
        'parameters': {'type': 'object',
            'properties': {'timezone': {'type': 'string', 'description': 'IANA time zone, e.g. America/Los_Angeles or UTC.'}}},
    }},
    {'type': 'function', 'function': {
        'name': 'days_until_weekday',
        'description': 'Calculate how many days from today until the next occurrence of a given weekday (e.g. Thursday). Use this AFTER you know which day an event happens, to work out how far away it is.',
        'parameters': {'type': 'object',
            'properties': {'weekday': {'type': 'string', 'description': 'A weekday name, e.g. Monday, Thursday, Sunday.'}},
            'required': ['weekday']},
    }},
]

available_tools = {
    'search_campus_info': search_campus_info,
    'get_current_time': get_current_time,
    'days_until_weekday': days_until_weekday,
}

def run_tool(name, arguments):
    if name not in available_tools:
        return f"Tool '{name}' is not available."
    try:
        return available_tools[name](**arguments)
    except Exception as exc:
        return f"Tool '{name}' failed: {exc}"

## Step 3 — Streaming at its simplest (no tools, no memory)

Add `stream=True` and you no longer get one message back — you get an iterator of **chunks**. Each chunk carries a little `delta`. For plain text, the only field that matters is `delta.content`: print it as it arrives, and the answer types itself out.

(One guard: some chunks — like a trailing usage report — have no `choices`. Skip those.)

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'In two sentences, what is GPU acceleration?'}],
    stream=True,
)

for chunk in resp:
    if not chunk.choices:          # e.g. a trailing usage-only chunk
        continue
    print(chunk.choices[0].delta.content or '', end='', flush=True)
print()

## Step 4 — The catch: tool calls arrive in fragments

Text is easy. Tool calls are the part that surprises people. When the model decides to call a tool, the call does **not** arrive in one piece — the function name shows up in one chunk, and the arguments JSON dribbles in across several more. Each fragment is tagged with a `tc.index` so you know which call it belongs to (the model can request more than one).

So you keep a small dictionary keyed by index, and for each fragment you set the `id`/`name` and **concatenate** the `arguments` string. After the stream ends, each bucket holds one complete tool call, ready to run. That reassembly is the only genuinely new idea in this workshop — the loop around it is unchanged from Workshop 7.

## Step 5 — `ChatSession` with both `chat()` and `stream()`

`stream()` is the Workshop 7 loop with the accumulation from Steps 3–4 folded in: stream a turn, reassemble whatever came back, then either run the tools and loop, or stop because the final answer already streamed. `_trim()` (trim-by-turns) and the memory behavior are exactly as in Workshop 7.

In [ ]:
SYSTEM_PROMPT = (
    'You are a USC campus assistant having an ongoing conversation with a student. '
    'You remember everything said earlier in this conversation.\n\n'
    "When a question refers back to something already discussed — words like 'that', "
    "'those', 'then', 'it', or 'the second one' — resolve the reference from the "
    'conversation so far before doing anything else.\n\n'
    'You have three tools: search_campus_info (find USC facts), get_current_time '
    '(today\'s date and time), and days_until_weekday (days from today until a weekday). '
    'Work step by step: decide what you still need, call ONE tool, read the result, then '
    'continue. Before calling a tool, check whether the conversation ALREADY contains the '
    'fact you need — do not re-search for something you found a turn ago.\n\n'
    'To compare how soon two days are, call days_until_weekday for EACH day and compare the '
    'numbers it returns — never estimate the number of days yourself.\n\n'
    'Base every answer strictly on tool results and what was said in the conversation, never '
    'on your own assumptions about USC. If you cannot find the answer, reply exactly: '
    "I don't have that information — check with the USC AI Club."
)

MAX_STEPS = 5

class ChatSession:
    def __init__(self, max_turns=8, verbose=True):
        self.system = {'role': 'system', 'content': SYSTEM_PROMPT}
        self.messages = [self.system]
        self.max_turns = max_turns
        self.verbose = verbose

    def reset(self):
        self.messages = [self.system]

    def _trim(self):
        user_indices = [i for i, m in enumerate(self.messages) if m.get('role') == 'user']
        if len(user_indices) <= self.max_turns:
            return
        cut = user_indices[-self.max_turns]
        self.messages = [self.system] + self.messages[cut:]

    def chat(self, user_message):                 # Workshop 7: non-streaming
        self.messages.append({'role': 'user', 'content': user_message})
        for step in range(1, MAX_STEPS + 1):
            response = client.chat.completions.create(
                model=MODEL, messages=self.messages, tools=tools,
                tool_choice='auto', temperature=0.2, max_tokens=400)
            message = response.choices[0].message
            self.messages.append(message.model_dump(exclude_none=True))
            if not message.tool_calls:
                self._trim()
                return message.content
            for tool_call in message.tool_calls:
                try:
                    arguments = json.loads(tool_call.function.arguments or '{}')
                except json.JSONDecodeError:
                    arguments = {}
                result = run_tool(tool_call.function.name, arguments)
                if self.verbose:
                    print(f'  step {step} · {tool_call.function.name}({json.dumps(arguments)}) -> {result}')
                self.messages.append({'role': 'tool', 'tool_call_id': tool_call.id,
                                      'name': tool_call.function.name, 'content': str(result)})
        fallback = 'I reached the step limit before finishing — try asking a narrower question.'
        self.messages.append({'role': 'assistant', 'content': fallback})
        self._trim()
        return fallback

    def stream(self, user_message):               # Workshop 8: streaming
        self.messages.append({'role': 'user', 'content': user_message})
        for step in range(1, MAX_STEPS + 1):
            stream_resp = client.chat.completions.create(
                model=MODEL, messages=self.messages, tools=tools,
                tool_choice='auto', temperature=0.2, max_tokens=400, stream=True)

            text_parts = []
            tool_fragments = {}          # index -> {'id', 'name', 'arguments'}
            header_printed = False

            for chunk in stream_resp:
                if not chunk.choices:
                    continue
                delta = chunk.choices[0].delta
                if delta.content:                  # a piece of the visible answer
                    if not header_printed:
                        print('Assistant: ', end='', flush=True)
                        header_printed = True
                    print(delta.content, end='', flush=True)
                    text_parts.append(delta.content)
                for tc in (delta.tool_calls or []):   # a fragment of a tool call
                    slot = tool_fragments.setdefault(tc.index, {'id': '', 'name': '', 'arguments': ''})
                    if tc.id and not slot['id']:
                        slot['id'] = tc.id
                    if tc.function and tc.function.name and not slot['name']:
                        slot['name'] = tc.function.name
                    if tc.function and tc.function.arguments:
                        slot['arguments'] += tc.function.arguments

            if header_printed:
                print()

            text = ''.join(text_parts)
            tool_calls = [tool_fragments[i] for i in sorted(tool_fragments)]

            assistant_msg = {'role': 'assistant'}
            if tool_calls:
                assistant_msg['tool_calls'] = [
                    {'id': tc['id'], 'type': 'function',
                     'function': {'name': tc['name'], 'arguments': tc['arguments']}}
                    for tc in tool_calls]
                if text:
                    assistant_msg['content'] = text
            else:
                assistant_msg['content'] = text
            self.messages.append(assistant_msg)

            if not tool_calls:                     # final answer already streamed
                self._trim()
                return text

            for tc in tool_calls:                  # run tools, then loop and stream again
                try:
                    arguments = json.loads(tc['arguments'] or '{}')
                except json.JSONDecodeError:
                    arguments = {}
                result = run_tool(tc['name'], arguments)
                if self.verbose:
                    print(f"  step {step} · {tc['name']}({json.dumps(arguments)}) -> {result}")
                self.messages.append({'role': 'tool', 'tool_call_id': tc['id'],
                                      'name': tc['name'], 'content': str(result)})

        fallback = 'I reached the step limit before finishing — try asking a narrower question.'
        print(f'Assistant: {fallback}')
        self.messages.append({'role': 'assistant', 'content': fallback})
        self._trim()
        return fallback

## Step 6 — Run it: feel the difference

First a non-streaming `chat()` call (Workshop 7) — the answer appears all at once after a pause. Then the same session keeps going with `stream()` — the tool steps print, then the answer types itself out.

In [ ]:
print('── Without streaming (answer arrives all at once) ──')
session = ChatSession(verbose=True)
print('You:       What are the USC GPU lab hours?')
print(f"Assistant: {session.chat('What are the USC GPU lab hours?')}")

print('\n── With streaming (watch the answer appear token by token) ──')
for user_message in [
    'When does the USC AI Club meet?',                          # tool call, then streams
    'How many days until that?',                                # memory + tool, then streams
    'Which is sooner, that meeting or the AI/ML office hours?',  # multi-step, then streams
]:
    print(f'\nYou:       {user_message}')
    session.stream(user_message)

## What just happened

- The **non-streaming** call pauses, then prints the whole answer — the Workshop 7 feel.
- The **streaming** calls print each token as it's generated. On the tool turns you see the `step` line first (the model decided to call a tool), then the final answer streams out.
- Memory still works — *"how many days until that?"* resolves *"that"* to Thursday from the earlier turn — and so does multi-step reasoning. Streaming changed how the answer is *delivered*, not how the agent *thinks*.

Two things worth knowing:

- **Tool calls stream as fragments.** Reassembling them by `index` is the one new idea here. Text is the easy case; tool calls are why a streaming agent is more than `stream=True`.
- **One call per turn, low time-to-first-token.** We stream the same call that decides on tools — so the first token appears as soon as the model starts talking, which is the entire point. (Don't be tempted to do a non-streaming call first to "check for tools" and then re-stream — that makes the first token *slower* than not streaming at all.)

## The full series

- **Workshop 1:** First API call.
- **Workshop 2:** Embedding-based RAG.
- **Workshop 3:** Guardrails.
- **Workshop 4:** Run NIM on your own GPU.
- **Workshop 5:** Tool calling — one tool.
- **Workshop 6:** Multi-step ReAct — chaining tools.
- **Workshop 7:** Memory — multi-turn conversation.
- **Workshop 8 (this notebook):** Streaming — token-by-token, real-time output.

Star and fork the repo for your school: https://github.com/torkian/nvidia-nim-workshop